<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎭 DramaBox - Expressive TTS</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Emotional dialogue, laughs, whispers & voice cloning | FP16 DiT running on both T4s | Powered by Resemble AI DramaBox + Gemma-3 12B</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is DramaBox?

**DramaBox** by Resemble AI turns a written scene into expressive speech: shouting, whispering, laughing, sighing, crying, with the voice of any 10-second reference clip. You write the scene directions outside the quotes and the dialogue inside them.

| Feature | Details |
|----|----|
| Model | DramaBox v1.0: the LTX-2.3 audio DiT (3.2B) fine-tuned for expressive speech, plus the LTX audio VAE and a 48 kHz vocoder |
| Text encoder | Gemma-3 12B instruct, 4-bit (bitsandbytes) |
| Voices | 8 built-in voices, or clone any voice from a 10+ second clip (upload or record) |
| Long texts | Split at sentence ends, generated with the same voice + seed, joined with a crossfade |
| Output | 48 kHz stereo WAV with the Resemble Perth watermark, saved to `/kaggle/working/outputs` (Kaggle Output panel) |

### ⚡ What makes this version fast

| Change | Why it helps on T4 |
|----|----|
| **FP16 DiT** instead of BF16 | T4 has no BF16 tensor cores, so BF16 ran on a slow emulated path |
| **Both GPUs denoise at once** | GPU 0 runs the prompt pass while GPU 1 runs the negative-prompt (CFG) pass on its own copy of the DiT |
| **STG shares blocks 0-28** | STG only changes block 29, so it branches off the prompt pass there instead of re-running 29 identical blocks |
| **Gemma-3 on real tokens only** | DramaBox padded every prompt to 1024 tokens. Running only the real tokens gives the identical result with ~10x less text-encoder work |
| **Caching** | The negative prompt is encoded once; prompts and voice references are cached, so a new seed skips both encoders |

The method is DramaBox's own: same guidance (CFG + STG + rescale), sampler, voice conditioning and decoding. What changed is the number format (FP16 DiT, FP32 latents and VAE) and how the work is spread over the two GPUs.

### Quick Start
1. **Settings → Accelerator → GPU T4 x2**, and **turn on Internet** in the Settings sidebar
2. Run all cells. **Step 1** installs and downloads ~16 GB of weights. **Step 2** loads the model and starts the UI
3. Open the **Cloudflare** link or the **Gradio** share link printed at the bottom of Step 2
4. Pick a **voice** (or upload your own 10+ s clip), write a scene, press **Generate Speech**
5. **Prompt tips:** dialogue inside `"double quotes"`, directions outside. Sounds like `"Hahaha"` or `"Mmmm"` go inside the quotes; actions like *She sighs.* go outside

> 💡 **Save the downloads:** after the first run, create a Kaggle Dataset from `/kaggle/tmp/dramabox_models` and attach it with **Add Input**. Step 1 finds the big files there and skips the download.

---
## ⚙️ Step 1: Setup + model downloads

Fetches the DramaBox code (pinned to a tested commit) and the Python packages, then downloads the weights to `/kaggle/tmp/dramabox_models` with one progress bar:
- **DramaBox DiT** (6.6 GB) + **audio VAE / vocoder** (1.9 GB)
- **Gemma-3 12B 4-bit** text encoder (7.8 GB)

In [ ]:
# Step 1: Environment, pinned DramaBox code, dependencies and model downloads (~16 GB)
import os
import sys
import time
import json
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~20 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_dramabox.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()
T_START = time.time()

# ---- Settings ----
DRAMABOX_COMMIT = "a70a5818e103c1c9fef22409c1e0c707ebf4f8a7"  # resemble-ai/DramaBox, 23 May 2026
PERTH_COMMIT = "ff1c8ac55a976971245cdd53c18d6131ca00d993"     # resemble-ai/Perth (audio watermark)
ON_KAGGLE = os.path.isdir("/kaggle")
REPO_DIR = "/kaggle/tmp/DramaBox" if ON_KAGGLE else os.path.abspath("DramaBox")
MODELS_DIR = "/kaggle/tmp/dramabox_models" if ON_KAGGLE else os.path.abspath("dramabox_models")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no second copy of the weights in the xet cache
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("=" * 70)
print("🎭 DramaBox - Expressive TTS")
print("📺 Created by: AIQUEST Academy | YouTube & X: @AIQuestAcademy")
print("=" * 70)
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip().splitlines()
    for i, line in enumerate(smi):
        print(f"✅ GPU {i}: {line}")
    if len(smi) < 2:
        print("⚠️ One GPU only: it works, but slower. For full speed use Settings → Accelerator → GPU T4 x2")
except Exception:
    print("❌ No GPU found: Settings → Accelerator → GPU T4 x2")


def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def run_quiet(label, cmd):
    """Run a command with one live spinner line; show its output only if it fails."""
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    reader = threading.Thread(target=lambda: lines.extend(proc.stdout), daemon=True)
    reader.start()
    frames = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"
    while proc.poll() is None:
        sys.stdout.write(f"\r  {frames[int(time.time() * 4) % 10]} {label} | {fmt_time(time.time() - t0)}".ljust(80))
        sys.stdout.flush()
        time.sleep(0.25)
    reader.join(timeout=5)
    sys.stdout.write("\r" + " " * 80 + "\r")
    if proc.returncode != 0:
        print(f"  ❌ {label} failed:\n" + "".join(lines[-40:]))
        raise SystemExit(f"❌ {label} failed (see above)")
    print(f"  ✅ {label} ({fmt_time(time.time() - t0)})")


# ---- 1. DramaBox code, pinned to a tested commit ----
print("\n🧩 Code and dependencies")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "init", "-q", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "add", "origin", "https://github.com/resemble-ai/DramaBox.git"], check=True)
run_quiet(f"DramaBox @ {DRAMABOX_COMMIT[:10]}", ["git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", DRAMABOX_COMMIT])
subprocess.run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD"], check=True)  # also undoes stale edits

# ---- 2. Dependencies (Kaggle's own torch / torchaudio stay) ----
run_quiet("Python packages", [
    sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5",
    "transformers==4.54.0", "accelerate>=1.1.1", "bitsandbytes>=0.46.0", "gradio==5.29.0", "huggingface_hub[hf_xet]",
    "safetensors>=0.4.3", "einops>=0.7.0", "av>=12.0.0", "sentencepiece", "soundfile", "PyYAML",
    f"resemble-perth @ git+https://github.com/resemble-ai/Perth.git@{PERTH_COMMIT}"])

# ---- 3. Model weights (parallel; big files from an attached Kaggle Dataset are reused) ----
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
DRAMABOX_REPO, GEMMA_REPO = "ResembleAI/Dramabox", "unsloth/gemma-3-12b-it-bnb-4bit"
gemma_files = [f for f in api.list_repo_files(GEMMA_REPO) if f not in (".gitattributes", "README.md")]
FILES = [(DRAMABOX_REPO, "dramabox-dit-v1.safetensors", "dramabox"),
         (DRAMABOX_REPO, "dramabox-audio-components.safetensors", "dramabox")] + \
        [(GEMMA_REPO, f, "gemma-3-12b-it-bnb-4bit") for f in gemma_files]
SIZES = {}
for repo in (DRAMABOX_REPO, GEMMA_REPO):
    for info in api.get_paths_info(repo, [f for r, f, _ in FILES if r == repo]):
        SIZES[(repo, info.path)] = getattr(info, "size", 0) or 0

attached = {}
if os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            path = os.path.join(root, f)
            try:
                attached.setdefault((f, os.path.getsize(path)), path)
            except OSError:
                pass


def local_path(item):
    repo, filename, sub = item
    return os.path.join(MODELS_DIR, sub, filename)


def fetch(item):
    repo, filename, sub = item
    dest = local_path(item)
    if os.path.exists(dest):
        return f"✓ {filename}"
    size = SIZES.get((repo, filename), 0)
    source = attached.get((os.path.basename(filename), size)) if size > 50e6 else None
    if source:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        os.symlink(source, dest)
        return f"✓ {filename} (from attached dataset)"
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=repo, filename=filename, local_dir=os.path.join(MODELS_DIR, sub))
            return f"✅ {filename}" + (f" ({size / 1e9:.1f} GB)" if size > 50e6 else "")
        except Exception as e:
            if attempt == 3:
                raise
            print(f"  ⚠️ {filename}: {e} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)


def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total


os.makedirs(MODELS_DIR, exist_ok=True)
todo = [item for item in FILES if not os.path.exists(local_path(item))]
need = sum(SIZES.get((r, f), 0) for r, f, _ in todo)
free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
print(f"\n🚀 Downloading {need / 1e9:.1f} GB of weights to {MODELS_DIR} ({free_gb:.0f} GB free)...")
if need / 1024**3 > free_gb - 2:
    raise SystemExit(f"❌ Not enough disk space: need {need / 1e9:.0f} GB, {free_gb:.0f} GB free.")
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]


def say(message):  # a normal line printed above the live bar
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)


def monitor():
    # One live line: GB done / total, %, smoothed speed, ETA. hf_xet writes into partial files, so bytes on disk
    # (minus what was already there) is the real progress.
    base = folder_bytes(MODELS_DIR)
    total = max(need, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"  ⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True


threading.Thread(target=monitor, daemon=True).start()
t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=6) as pool:
        jobs = [pool.submit(fetch, item) for item in sorted(FILES, key=lambda it: -SIZES.get((it[0], it[1]), 0))]
        for job in jobs:
            result = job.result()
            if "GB" in result or "attached" in result:  # only the big files get a line
                say("  " + result)
finally:
    done.set()
    time.sleep(1.1)
say(f"  ✅ {need / 1e9:.1f} GB downloaded in {fmt_time(time.time() - t0)}"
    + (f" (avg {need / 1e6 / max(time.time() - t0, 1):.0f} MB/s)" if need else ""))
for sub in ("dramabox", "gemma-3-12b-it-bnb-4bit"):
    shutil.rmtree(os.path.join(MODELS_DIR, sub, ".cache"), ignore_errors=True)

# Settings for Step 2's engine (saved to a file, so Step 2 can be re-run alone after a kernel restart)
SETTINGS = {"DRAMABOX_REPO": REPO_DIR, "DRAMABOX_MODELS": MODELS_DIR}
os.environ.update(SETTINGS)
with open("dramabox_settings.json", "w") as f:
    json.dump(SETTINGS, f)
print("\n" + "=" * 70)
print(f"✅ Step 1 done in {fmt_time(time.time() - T_START)} | disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB")
print("=" * 70)

---
## 🚀 Step 2: Start the engine + UI

Loads everything once and keeps it in VRAM:
- **GPU 0:** DiT (FP16), embeddings connector, audio VAE + vocoder, watermarker
- **GPU 1:** Gemma-3 12B 4-bit + a second DiT copy for the negative-prompt pass

The cell shows one progress line per loading stage, then the **Cloudflare** and **Gradio** links. Every generation prints a short summary here: steps, seconds per step, and how much faster than real time it ran.

- The page shows a live progress bar; reloading the page or a dropped connection does not stop the job
- **Stop** ends the job after the current step. To restart the engine, re-run this cell
- The newest audio is also at `<link>/aiq_audio`

In [ ]:
# Step 2: Fast DramaBox engine + branded Gradio UI
# Writes run_dramabox.py, then runs it: model load (~2-3 min) → public links.
import codecs
import json
import os
import re
import subprocess
import sys
import time

RUN_SCRIPT_CODE = r'''"""DramaBox expressive TTS engine for Kaggle GPU T4 x2 - AIQUEST Academy.

Step 2 of the notebook writes this file as run_dramabox.py and runs it. It drives the official DramaBox code
(resemble-ai/DramaBox, pinned in Step 1) with these speed changes:

* FP16 DiT. T4 (Turing) has no BF16 tensor cores, so the 3.2B DiT in BF16 ran on a slow emulated path. FP16 runs on the
  tensor cores. The latent state, the guidance math and the Euler steps stay in FP32.
* Both GPUs denoise at once. GPU 0 runs the prompt pass, GPU 1 runs the negative-prompt (CFG) pass on its own copy of
  the DiT. One thread drives both block by block, so neither GPU waits for the other.
* The STG pass shares blocks 0-28 with the prompt pass. STG only skips self-attention in block 29, so both passes are
  identical until then: the prompt pass's hidden state is copied at block 29 and both finish as a batch of 2.
  Per step that is 29 + 2 x 19 block-runs on GPU 0 instead of 3 x 48 on one GPU.
* Gemma-3 runs on the prompt's real tokens only, not on 1024 padded tokens. The embeddings connector moves the real
  tokens to the front and fills the rest with learnable registers, so the padded positions never reach the DiT: the
  result is the same, with ~10x less Gemma work. The negative prompt is encoded once at startup.
* Prompt encodings and voice-reference latents are cached, so a new seed or new settings skip both encoders.
* Gemma's vision tower is dropped and its 2 GB token-embedding table lives in CPU RAM: that frees the room on GPU 1 for
  the second DiT copy.
* Audio VAE + vocoder run in FP32 (Turing has no BF16 convolution kernels).
"""
import copy
import dataclasses
import gc
import glob
import json
import logging
import os
import random
import re
import subprocess
import sys
import threading
import time
import urllib.request
import warnings
from collections import OrderedDict

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)  # the DramaBox modules call basicConfig(INFO) on import: this one wins

REPO_DIR = os.environ.get("DRAMABOX_REPO", "/kaggle/tmp/DramaBox")
MODELS_DIR = os.environ.get("DRAMABOX_MODELS", "/kaggle/tmp/dramabox_models")
WORK_DIR = os.environ.get("DRAMABOX_WORK_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
DIT_FILE = os.path.join(MODELS_DIR, "dramabox", "dramabox-dit-v1.safetensors")
COMPONENTS_FILE = os.path.join(MODELS_DIR, "dramabox", "dramabox-audio-components.safetensors")
GEMMA_DIR = os.path.join(MODELS_DIR, "gemma-3-12b-it-bnb-4bit")
VOICES_DIR = os.path.join(REPO_DIR, "assets", "voices")
SERVER_PORT = 7860
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
for _p in (os.path.join(REPO_DIR, "ltx2"), os.path.join(REPO_DIR, "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import psutil
import soundfile as sf
import torch

import gradio as gr

DIT_DTYPE = torch.float16  # DiT weights + activations (T4 tensor cores)
EP_DTYPE = torch.bfloat16  # embeddings connector, as stored in the checkpoint


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _fmt_secs(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


class ConsoleBar:
    """One live progress line per phase, redrawn every second with a carriage return (Kaggle's cell output overwrites
    it in place). A finished phase is replaced by a one-line summary. Step-based phases interpolate inside the current
    step from the previous step's duration, so the bar keeps moving and the ETA counts down between steps."""

    WIDTH, LINE = 24, 104

    def __init__(self, stream):
        self.out = stream
        self.lock = threading.RLock()
        self.phase = None
        self.visible = False
        self.text_pending = False  # another writer is in the middle of a line: don't draw over it
        threading.Thread(target=self._loop, daemon=True).start()

    def start(self, key, label, total=None, unit=""):
        with self.lock:
            self.finish()
            now = time.time()
            self.phase = dict(key=key, label=label, total=total, unit=unit, done=0, t0=now, t_done=now, dt=None)
            self._draw()

    def update(self, done, total=None):
        with self.lock:
            p = self.phase
            if p is None:
                return
            now = time.time()
            if total:
                p["total"] = total
            if done > p["done"]:
                p["dt"] = (now - p["t_done"]) / (done - p["done"])
                p["t_done"], p["done"] = now, done
            self._draw()

    def finish(self, note=""):
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            elapsed = time.time() - p["t0"]
            line = f"  ✅ {p['label']}"
            if p["total"]:
                line += f" {p['done']}/{p['total']} {p['unit']}"
            line += f" in {_fmt_secs(elapsed) if elapsed >= 1 else f'{elapsed:.1f}s'}"
            if p["unit"] == "steps" and p["done"]:
                line += f" ({elapsed / p['done']:.2f}s/step)"
            if note:
                line += f" {note}"
            self._clear()
            self.out.write(line + "\n")
            self.out.flush()

    def abort(self, why="failed"):
        """End the live line of a phase that did not complete (no ✅ summary)."""
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            self._clear()
            self.out.write(f"  ❌ {p['label']} {why} after {_fmt_secs(time.time() - p['t0'])}\n")
            self.out.flush()

    def _render(self):
        p, now = self.phase, time.time()
        elapsed = now - p["t0"]
        if not p["total"]:
            spinner = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"[int(now * 4) % 10]
            return f"  {spinner} {p['label']} | {_fmt_secs(elapsed)}"
        progress = p["done"]
        if p["dt"] and p["done"] < p["total"]:
            progress += min(0.97, (now - p["t_done"]) / p["dt"])
        frac = min(1.0, progress / p["total"])
        fill = int(round(frac * self.WIDTH))
        line = (f"  ⏳ {p['label']:<18} [{'█' * fill}{'░' * (self.WIDTH - fill)}] {frac * 100:3.0f}% | "
                f"{p['done']}/{p['total']} {p['unit']}")
        if p["dt"]:
            eta = max(0.0, (p["total"] - p["done"]) * p["dt"] - (now - p["t_done"]))
            line += f" | {p['dt']:.2f}s each | ETA {_fmt_secs(eta)}"
        return line + f" | {_fmt_secs(elapsed)}"

    def _draw(self):
        if self.phase is None or self.text_pending:
            return
        self.out.write("\r" + self._render().ljust(self.LINE))
        self.out.flush()
        self.visible = True

    def _clear(self):
        if self.visible:
            self.out.write("\r" + " " * self.LINE + "\r")
            self.visible = False

    def _loop(self):
        while True:
            time.sleep(1.0)
            with self.lock:
                self._draw()


class _BarAwareStream:
    """stdout / stderr wrapper: ordinary log lines clear the live bar first and print above it."""

    def __init__(self, stream, bar):
        self._stream, self._bar = stream, bar

    def write(self, data):
        with self._bar.lock:
            if data:
                self._bar._clear()
                self._bar.text_pending = not data.endswith("\n")
            return self._stream.write(data)

    def flush(self):
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


BAR = ConsoleBar(sys.stdout)
sys.stdout = _BarAwareStream(sys.stdout, BAR)
sys.stderr = _BarAwareStream(sys.stderr, BAR)

if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU found. Settings → Accelerator → GPU T4 x2")

N_GPU = torch.cuda.device_count()
DEV0 = torch.device("cuda", 0)                           # DiT (prompt + STG passes), embeddings connector, VAE, vocoder
DEV1 = torch.device("cuda", 1) if N_GPU > 1 else DEV0    # Gemma-3 + the DiT copy for the negative-prompt pass
torch.cuda.set_device(DEV0)

torch.backends.cuda.enable_flash_sdp(False)  # Turing (SM 7.5) has no flash kernel; SDPA uses memory-efficient attention
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
# FP16 GEMMs keep FP32 accumulation all the way (no FP16 split-K reductions): no overflow on the 8192-wide FFN.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False

_props = [torch.cuda.get_device_properties(i) for i in range(N_GPU)]
print("=" * 70)
print("🎭 DramaBox - Expressive TTS | AIQUEST Academy")
print(f"⚡ GPUs: {N_GPU} x {_props[0].name} ({_gb(_props[0].total_memory)} each, SM {_props[0].major}.{_props[0].minor}) | "
      f"RAM: {_gb(psutil.virtual_memory().total)} ({_gb(psutil.virtual_memory().available)} free)")
if N_GPU < 2:
    print("⚠️ Only one GPU: everything runs on GPU 0 (slower). Use Settings → Accelerator → GPU T4 x2 for full speed.")
print("=" * 70)
sys.stdout.flush()

# ==== DramaBox / LTX-2 code (from the pinned repo) ====
import transformers
transformers.logging.set_verbosity_error()

from audio_conditioning import AudioConditionByReferenceLatent
from duration_estimator import estimate_speech_duration
from inference_server import DEFAULT_NEG, _equal_power_crossfade, auto_rescale_for_cfg, estimate_duration
from text_chunker import chunk_prompt_for_duration
from ltx_core.components.diffusion_steps import EulerDiffusionStep
from ltx_core.components.guiders import MultiModalGuider, MultiModalGuiderParams
from ltx_core.components.noisers import GaussianNoiser
from ltx_core.components.patchifiers import AudioPatchifier
from ltx_core.components.schedulers import LTX2Scheduler
from ltx_core.guidance.perturbations import BatchedPerturbationConfig, Perturbation, PerturbationConfig, PerturbationType
from ltx_core.loader.registry import DummyRegistry
from ltx_core.loader.sd_ops import SDOps
from ltx_core.loader.single_gpu_model_builder import SingleGPUModelBuilder as Builder
from ltx_core.model.audio_vae import decode_audio as vae_decode_audio
from ltx_core.model.audio_vae import encode_audio as vae_encode_audio
from ltx_core.model.model_protocol import ModelConfigurator
from ltx_core.model.transformer.attention import AttentionFunction
from ltx_core.model.transformer.modality import Modality
from ltx_core.model.transformer.model import LTXModel, LTXModelType
from ltx_core.model.transformer.rope import LTXRopeType
from ltx_core.model.transformer.text_projection import create_caption_projection
from ltx_core.model.transformer.transformer_args import TransformerArgs
from ltx_core.text_encoders.gemma.embeddings_processor import convert_to_additive_mask
from ltx_core.tools import AudioLatentTools
from ltx_core.types import Audio, AudioLatentShape, VideoPixelShape
from ltx_pipelines.utils.blocks import AudioConditioner, AudioDecoder, PromptEncoder
from ltx_pipelines.utils.helpers import timesteps_from_mask
from ltx_pipelines.utils.media_io import decode_audio_from_file
from ltx_pipelines.utils.samplers import _step_state

logging.getLogger().setLevel(logging.ERROR)
for _name in ("httpx", "gradio", "urllib3", "bitsandbytes", "transformers", "accelerate"):
    logging.getLogger(_name).setLevel(logging.ERROR)

T_LOAD = time.time()

# ==== 1. Gemma-3 12B (4-bit) + embeddings connector ====
BAR.start("gemma", "Gemma-3 12B 4-bit")
with torch.cuda.device(DEV1):
    _prompt_encoder = PromptEncoder(checkpoint_path=COMPONENTS_FILE, gemma_root=GEMMA_DIR, dtype=torch.bfloat16,
                                    device=DEV1, warm=True, use_bnb_4bit=True, audio_only=True)
TEXT_ENCODER = _prompt_encoder._warm_text_encoder
EP = _prompt_encoder._warm_embeddings_processor
del _prompt_encoder

_hf = TEXT_ENCODER.model
if hasattr(getattr(_hf, "model", None), "language_model"):  # transformers >= 4.52: model.language_model
    _core = _hf.model
    LM = _core.language_model
else:                                                       # older layout: language_model is a CausalLM
    _core = _hf
    LM = getattr(_hf.language_model, "model", _hf.language_model)
for _name in ("vision_tower", "multi_modal_projector"):  # text only: the image path is never used
    if getattr(_core, _name, None) is not None:
        setattr(_core, _name, None)
EMBED = LM.embed_tokens.to("cpu")  # 262k x 3840 table (2 GB): a lookup, done on the CPU
TOKENIZER = TEXT_ENCODER.tokenizer.tokenizer
MAX_TOKENS = 1024  # DramaBox pads every prompt to 1024 tokens; the connector output keeps that length
EP = EP.to(DEV0).eval()
FEATURES = EP.feature_extractor
CONNECTOR = EP.audio_connector
gc.collect()
with torch.cuda.device(DEV1):
    torch.cuda.empty_cache()
BAR.finish()


# Turing has no BF16 tensor cores, so a BF16 GEMM can be slower than plain FP32. Gemma-3 overflows in FP16, so the
# choice is BF16 or FP32: measure both on a real Gemma layer shape and keep the faster one.
def _gemm_seconds(dtype, m=256, k=3840, n=15360, reps=3):
    a = torch.randn(m, k, device=DEV1, dtype=dtype)
    b = torch.randn(n, k, device=DEV1, dtype=dtype)
    torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(DEV1)
    t = time.time()
    for _ in range(reps):
        torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(DEV1)
    return (time.time() - t) / reps


try:
    _bf16_s = _gemm_seconds(torch.bfloat16)
except Exception:
    _bf16_s = float("inf")
try:
    _fp32_s = _gemm_seconds(torch.float32)
except Exception:
    _fp32_s = float("inf")
GEMMA_DTYPE = torch.float32 if _fp32_s < _bf16_s else torch.bfloat16
with torch.cuda.device(DEV1):
    torch.cuda.empty_cache()


def _set_gemma_dtype(dtype):
    import bitsandbytes as bnb
    for module in LM.modules():
        if isinstance(module, bnb.nn.Linear4bit):
            module.compute_dtype = dtype


_set_gemma_dtype(GEMMA_DTYPE)

CTX_CACHE = OrderedDict()  # prompt text -> FP16 audio context [1, 1024, 2048] on GPU 0


@torch.inference_mode()
def encode_text(text):
    """Gemma-3 on the real tokens only, then DramaBox's feature extractor + audio connector on a 1024-long sequence.
    The connector compacts the real tokens to the front and fills the rest with learnable registers, so the padded
    positions (zeros here) are never read: same output as the original left-padded 1024-token run."""
    key = text.strip()
    if key in CTX_CACHE:
        CTX_CACHE.move_to_end(key)
        return CTX_CACHE[key]
    ids = TOKENIZER(key, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").input_ids
    n = ids.shape[1]
    with torch.cuda.device(DEV1):
        embeds = EMBED(ids).to(DEV1, GEMMA_DTYPE)
        # The real tokens keep the positions they had at the end of the left-padded 1024 sequence: RoPE is relative,
        # but its angles are computed in FP32, so the same absolute positions reproduce the original bit for bit.
        out = LM(inputs_embeds=embeds, attention_mask=torch.ones(1, n, dtype=torch.long, device=DEV1),
                 position_ids=torch.arange(MAX_TOKENS - n, MAX_TOKENS, device=DEV1)[None],
                 output_hidden_states=True, use_cache=False)
        # Per-token RMS norm of every layer (FeatureExtractorV2), one layer at a time to keep memory low
        normed = []
        for h in out.hidden_states[-(LM.config.num_hidden_layers + 1):]:  # embeddings + every layer
            h = h.float()
            normed.append((h * torch.rsqrt(h.pow(2).mean(dim=-1, keepdim=True) + 1e-6)).to(EP_DTYPE))
        del out
        normed = torch.stack(normed, dim=-1).reshape(1, n, -1)  # [1, n, 3840 * 49], same layout as the original
    normed = normed.to(DEV0)
    agg = FEATURES.audio_aggregate_embed
    real = agg(normed * (agg.out_features / FEATURES.embedding_dim) ** 0.5)  # _rescale_norm + aggregate projection
    feats = torch.zeros(1, MAX_TOKENS, agg.out_features, device=DEV0, dtype=real.dtype)
    feats[:, MAX_TOKENS - n:] = real  # left padding, as the tokenizer did
    mask = torch.zeros(1, MAX_TOKENS, dtype=torch.long, device=DEV0)
    mask[:, MAX_TOKENS - n:] = 1
    ctx, _ = CONNECTOR(feats, convert_to_additive_mask(mask, feats.dtype))
    ctx = ctx.to(DIT_DTYPE)  # the connector output is RMS-normalized: safe in FP16
    CTX_CACHE[key] = ctx
    while len(CTX_CACHE) > 32:
        CTX_CACHE.popitem(last=False)
    return ctx


BAR.start("neg", "Negative prompt")
try:
    NEG_CTX0 = encode_text(DEFAULT_NEG)
except Exception:
    if GEMMA_DTYPE == torch.bfloat16:
        raise
    GEMMA_DTYPE = torch.bfloat16  # FP32 compute failed on this setup: back to the checkpoint's BF16
    _set_gemma_dtype(GEMMA_DTYPE)
    NEG_CTX0 = encode_text(DEFAULT_NEG)
NEG_CTX1 = NEG_CTX0.to(DEV1)
BAR.finish(f"(Gemma GEMMs in {'FP32' if GEMMA_DTYPE == torch.float32 else 'BF16'}: "
           f"BF16 {_bf16_s * 1000:.0f} ms vs FP32 {_fp32_s * 1000:.0f} ms per layer)")


# ==== 2. DiT: FP16 copies on both GPUs ====
class AudioOnlyConfigurator(ModelConfigurator[LTXModel]):
    """Same audio-only LTX-2.3 model the DramaBox server builds."""

    @classmethod
    def from_config(cls, cfg):
        t = cfg.get("transformer", {})
        cp = None
        if not t.get("caption_proj_before_connector", False):
            with torch.device("meta"):
                cp = create_caption_projection(t, audio=True)
        return LTXModel(
            model_type=LTXModelType.AudioOnly,
            audio_num_attention_heads=t.get("audio_num_attention_heads", 32),
            audio_attention_head_dim=t.get("audio_attention_head_dim", 64),
            audio_in_channels=t.get("audio_in_channels", 128),
            audio_out_channels=t.get("audio_out_channels", 128),
            num_layers=t.get("num_layers", 48),
            audio_cross_attention_dim=t.get("audio_cross_attention_dim", 2048),
            norm_eps=t.get("norm_eps", 1e-6),
            attention_type=AttentionFunction(t.get("attention_type", "default")),
            positional_embedding_theta=10000.0,
            audio_positional_embedding_max_pos=[20.0],
            timestep_scale_multiplier=t.get("timestep_scale_multiplier", 1000),
            use_middle_indices_grid=t.get("use_middle_indices_grid", True),
            rope_type=LTXRopeType(t.get("rope_type", "interleaved")),
            double_precision_rope=t.get("frequencies_precision", False) == "float64",
            apply_gated_attention=t.get("apply_gated_attention", False),
            audio_caption_projection=cp,
            cross_attention_adaln=t.get("cross_attention_adaln", False),
        )


BAR.start("dit", "DiT 3.2B → FP16")
_builder = Builder(
    model_path=DIT_FILE,
    model_class_configurator=AudioOnlyConfigurator,
    model_sd_ops=SDOps("AO").with_matching(prefix="model.diffusion_model.").with_replacement("model.diffusion_model.", ""),
    registry=DummyRegistry(),
)
_dit_cpu = _builder.build(device=torch.device("cpu"), dtype=DIT_DTYPE).eval()  # converted in RAM, not in VRAM
gc.collect()
DIT1 = None
if N_GPU > 1:
    try:
        DIT1 = copy.deepcopy(_dit_cpu).to(DEV1)
    except torch.cuda.OutOfMemoryError:
        DIT1 = None
        gc.collect()
        with torch.cuda.device(DEV1):
            torch.cuda.empty_cache()
DIT0 = _dit_cpu.to(DEV0)
del _dit_cpu
gc.collect()
try:
    import ctypes
    ctypes.CDLL("libc.so.6").malloc_trim(0)  # hand the freed conversion buffers back to the OS
except Exception:
    pass
NUM_BLOCKS = len(DIT0.transformer_blocks)
STG_BLOCKS = [29]  # DramaBox's STG layer
BAR.finish("(GPU 0 + GPU 1)" if DIT1 is not None else "(GPU 0)")
if N_GPU > 1 and DIT1 is None:
    print("⚠️ The second DiT copy did not fit on GPU 1: all guidance passes run on GPU 0 (slower).")

# ==== 3. Audio VAE encoder / decoder + vocoder (FP32) and the Perth watermarker ====
BAR.start("vae", "Audio VAE + vocoder")
AUDIO_ENCODER = AudioConditioner(checkpoint_path=COMPONENTS_FILE, dtype=torch.float32, device=DEV0, warm=True)._warm_encoder
_decoder = AudioDecoder(checkpoint_path=COMPONENTS_FILE, dtype=torch.float32, device=DEV0, warm=True)
AUDIO_DECODER, VOCODER = _decoder._warm_decoder, _decoder._warm_vocoder
del _decoder
PATCHIFIER = AudioPatchifier(patch_size=1)
WATERMARKER = None
try:
    import perth
    try:
        WATERMARKER = perth.PerthImplicitWatermarker(device=str(DEV0))
    except Exception:
        WATERMARKER = perth.PerthImplicitWatermarker()
except Exception as e:
    print(f"⚠️ Perth watermarker unavailable ({e}); audio is saved without the watermark.")
BAR.finish()


# ==== Guided denoising across both GPUs ====
STG_PERTURBATION = PerturbationConfig([Perturbation(type=PerturbationType.SKIP_AUDIO_SELF_ATTN, blocks=STG_BLOCKS)])


def _append_copy_of_first(args):
    """Grow every batch-shaped tensor of a TransformerArgs by a copy of sample 0 (the prompt pass): the STG fork."""
    batch = args.x.shape[0]

    def grow(value):
        if torch.is_tensor(value) and value.dim() > 0 and value.shape[0] == batch:
            return torch.cat([value, value[:1]], dim=0)
        if isinstance(value, tuple):
            return tuple(grow(v) for v in value)
        return value

    return TransformerArgs(**{f.name: grow(getattr(args, f.name)) for f in dataclasses.fields(args)})


def _dit_pass(model, modality, fork_at):
    """Generator: one DiT forward, block by block (yields after each block, so one thread can keep two GPUs busy).
    With fork_at, sample 0 is duplicated at that block and the copy runs with STG's self-attention skip."""
    args = model.audio_args_preprocessor.prepare(modality, None)
    batch = args.x.shape[0]
    perturbations = BatchedPerturbationConfig.empty(batch)
    for idx, block in enumerate(model.transformer_blocks):
        if idx == fork_at:
            args = _append_copy_of_first(args)
            perturbations = BatchedPerturbationConfig([PerturbationConfig.empty()] * batch + [STG_PERTURBATION])
        _, args = block(video=None, audio=args, perturbations=perturbations)
        yield
    return model._process_output(model.audio_scale_shift_table, model.audio_norm_out, model.audio_proj_out,
                                 args.x, args.embedded_timestep)


def _run_together(*passes):
    """Advance several block generators in turn. CUDA launches are asynchronous, so the GPUs compute in parallel."""
    results, live = [None] * len(passes), list(range(len(passes)))
    while live:
        for k in list(live):
            try:
                next(passes[k])
            except StopIteration as stop:
                results[k] = stop.value
                live.remove(k)
    return results


class GuidedDiT:
    """x0 predictions for the prompt / negative-prompt / STG passes of one clip.
    Two GPUs: GPU 0 = prompt pass, forked into prompt + STG at block 29; GPU 1 = negative-prompt pass.
    One GPU: prompt + negative as a batch of 2, forked into a batch of 3 at block 29."""

    def __init__(self, state, pos_ctx, do_uncond, do_ptb):
        self.do_uncond, self.do_ptb = do_uncond, do_ptb
        self.fork_at = min(STG_BLOCKS) if do_ptb else None
        self.split = do_uncond and DIT1 is not None

        def static(device, n):  # per-clip constants, copied once instead of every step
            def rep(t):
                if t is None:
                    return None
                t = t.to(device)
                return t.repeat(n, *([1] * (t.dim() - 1))) if n > 1 else t
            return {"dmask": rep(state.denoise_mask), "pos": rep(state.positions), "amask": rep(state.attention_mask)}

        if self.split:
            self.main = (DIT0, DEV0, static(DEV0, 1), pos_ctx)
            self.side = (DIT1, DEV1, static(DEV1, 1), NEG_CTX1)
        else:
            ctx = torch.cat([pos_ctx, NEG_CTX0], dim=0) if do_uncond else pos_ctx
            self.main = (DIT0, DEV0, static(DEV0, ctx.shape[0]), ctx)
            self.side = None
        self.timestep_mask = state.denoise_mask  # [1, T, 1] on GPU 0

    @staticmethod
    def _modality(latent, sigma, device, s, ctx):
        n = ctx.shape[0]
        lat = latent.to(device=device, dtype=DIT_DTYPE)
        if n > 1:
            lat = lat.repeat(n, 1, 1)
        sigma = sigma.to(device)
        return Modality(latent=lat, sigma=sigma.expand(n), timesteps=timesteps_from_mask(s["dmask"], sigma),
                        positions=s["pos"], context=ctx, enabled=True, context_mask=None, attention_mask=s["amask"])

    def __call__(self, latent, sigma):
        model, dev, s, ctx = self.main
        passes = [_dit_pass(model, self._modality(latent, sigma, dev, s, ctx), self.fork_at)]
        if self.side is not None:
            model1, dev1, s1, ctx1 = self.side
            passes.append(_dit_pass(model1, self._modality(latent, sigma, dev1, s1, ctx1), None))
        outs = _run_together(*passes)
        v_main = outs[0].float()  # main order: prompt, [negative], [STG]
        cond = v_main[0:1]
        if self.split:
            uncond = outs[1].to(DEV0).float()
            ptb = v_main[1:2] if self.do_ptb else 0.0
        else:
            uncond = v_main[1:2] if self.do_uncond else 0.0
            ptb = v_main[-1:] if self.do_ptb else 0.0
        # velocity -> x0 in FP32 (to_denoised), per token: reference tokens have timestep 0
        ts = timesteps_from_mask(self.timestep_mask, sigma)
        x0 = lambda v: latent - v * ts  # noqa: E731
        return x0(cond), (x0(uncond) if torch.is_tensor(uncond) else 0.0), (x0(ptb) if torch.is_tensor(ptb) else 0.0)


class Cancelled(Exception):
    pass


CANCEL = threading.Event()


@torch.inference_mode()
def generate_clip(text, ref_latent, cfg, stg, duration_s, seed, steps, on_step=None):
    """One clip, same math as TTSServer.generate (DramaBox), with the speed changes described at the top."""
    fps = 25.0
    n_frames = int(round(duration_s * fps)) + 1
    n_frames = ((n_frames - 1 + 4) // 8) * 8 + 1
    target_shape = AudioLatentShape.from_video_pixel_shape(
        VideoPixelShape(batch=1, frames=n_frames, height=64, width=64, fps=fps))
    tools = AudioLatentTools(patchifier=PATCHIFIER, target_shape=target_shape)
    state = tools.create_initial_state(device=DEV0, dtype=torch.float32)
    if ref_latent is not None:
        state = AudioConditionByReferenceLatent(latent=ref_latent, strength=1.0).apply_to(state, tools)
    state = GaussianNoiser(generator=torch.Generator(device=DEV0).manual_seed(int(seed)))(state, noise_scale=1.0)

    pos_ctx = encode_text(text)
    guider = MultiModalGuider(params=MultiModalGuiderParams(
        cfg_scale=cfg, stg_scale=stg, stg_blocks=STG_BLOCKS, rescale_scale=auto_rescale_for_cfg(cfg), modality_scale=1.0))
    dit = GuidedDiT(state, pos_ctx, guider.do_unconditional_generation(), guider.do_perturbed_generation())
    sigmas = LTX2Scheduler().execute(steps=steps, latent=state.latent).to(DEV0)
    stepper = EulerDiffusionStep()
    for i in range(steps):
        if CANCEL.is_set():
            raise Cancelled()
        cond, uncond, ptb = dit(state.latent, sigmas[i])
        denoised = guider.calculate(cond, uncond, ptb, 0.0)
        state = _step_state(state, denoised, stepper, sigmas, i)
        torch.cuda.synchronize(DEV0)
        if on_step:
            on_step(i + 1, steps)

    state = tools.unpatchify(tools.clear_conditioning(state))
    latent = state.latent
    if latent.shape[2] > 513:  # DramaBox's end-of-clip silence-prior fix (frames 512-513 past 20 s)
        f0, f1 = 511, 514
        patched = latent.clone()
        for f in (512, 513):
            t = (f - f0) / (f1 - f0)
            patched[:, :, f, :] = (1.0 - t) * latent[:, :, f0, :] + t * latent[:, :, f1, :]
        latent = patched
    decoded = vae_decode_audio(latent.float(), AUDIO_DECODER, VOCODER)
    return decoded.waveform.float(), decoded.sampling_rate


REF_CACHE = OrderedDict()


@torch.inference_mode()
def encode_reference(path, ref_duration):
    """Voice reference -> VAE latent (cached per file + length). Same preprocessing as TTSServer.generate."""
    st = os.stat(path)
    key = (os.path.abspath(path), st.st_size, int(st.st_mtime), float(ref_duration))
    if key in REF_CACHE:
        REF_CACHE.move_to_end(key)
        return REF_CACHE[key]
    voice = decode_audio_from_file(path, DEV0, 0.0, ref_duration)
    if voice is None:
        raise ValueError("the voice reference has no audio stream")
    w = voice.waveform.float()
    if w.dim() == 2:
        if w.shape[0] == 1:
            w = w.repeat(2, 1)
        w = w.unsqueeze(0)
    elif w.dim() == 3 and w.shape[1] == 1:
        w = w.repeat(1, 2, 1)
    target_samples = int(ref_duration * voice.sampling_rate)
    if w.shape[-1] < target_samples:
        w = w.repeat(1, 1, (target_samples // w.shape[-1]) + 1)
    w = w[..., :target_samples]
    peak = w.abs().max()
    if peak > 0:
        w = w * (10 ** (-4.0 / 20) / peak)
    latent = vae_encode_audio(Audio(waveform=w, sampling_rate=voice.sampling_rate), AUDIO_ENCODER, None).float()
    REF_CACHE[key] = latent
    while len(REF_CACHE) > 8:
        REF_CACHE.popitem(last=False)
    return latent


# ==== 4. Warm-up: CUDA kernels on both GPUs, so the first real request runs at full speed ====
BAR.start("warmup", "Warm-up")
generate_clip('A man says calmly, "Hello there."', None, 2.5, 1.5, 3.0, 0, 2)
BAR.finish()
for _i in range(N_GPU):
    _free, _total = torch.cuda.mem_get_info(_i)
    print(f"  💾 GPU {_i}: {_gb(_total - _free)} / {_gb(_total)} in use")
print(f"✅ Engine ready in {_fmt_secs(time.time() - T_LOAD)}")
sys.stdout.flush()


# ==== Live progress (polled by the page over plain HTTP; the tunnels can hold back Gradio's event stream) ====
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0, "finished": 0.0, "audio": None, "message": ""}


def report(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


def preview_chunks(prompt, dur_mult, max_chunk_s, target_chunk_s):
    if not prompt or not prompt.strip():
        return ""
    raw = estimate_speech_duration(prompt) * float(dur_mult)
    if raw <= max_chunk_s:
        return f"📏 **Estimated length:** ~{raw:.1f} s · one clip (under the {max_chunk_s:.0f} s chunk cap)"
    chunks = chunk_prompt_for_duration(prompt, max_duration_s=float(max_chunk_s), target_duration_s=float(target_chunk_s),
                                       duration_multiplier=float(dur_mult))
    if len(chunks) == 1:
        return f"📏 **Estimated length:** ~{raw:.1f} s · one clip (no sentence end outside quotes to split at)"
    parts = " · ".join(f"{c.est_duration_s:.1f}s" for c in chunks)
    return (f"📏 **Estimated length:** ~{raw:.1f} s · **{len(chunks)} chunks** ({parts}), "
            f"joined with a crossfade")


def generate_audio(prompt, ref_path, steps, cfg, stg, dur_mult, gen_dur, ref_dur, seed, max_chunk, target_chunk,
                   crossfade_ms):
    """Plan chunks, generate each one, join, watermark, save. Returns (wav path, seed, status text)."""
    t0 = time.time()
    seed = random.randint(0, 2**31 - 1) if seed is None or int(seed) < 0 else int(seed)
    steps, cfg, stg = int(steps), float(cfg), float(stg)
    explicit = float(gen_dur or 0.0)
    est = explicit if explicit > 0 else estimate_duration(prompt, dur_mult)
    if est > max_chunk:  # long text: sentence-aligned chunks, same seed + voice, crossfaded (DramaBox's generate_long)
        chunks = chunk_prompt_for_duration(prompt, max_duration_s=float(max_chunk), target_duration_s=float(target_chunk),
                                           duration_multiplier=float(dur_mult))
        plan = [(c.text, estimate_duration(c.text, dur_mult)) for c in chunks]
    else:
        plan = [(prompt, est)]
    voice = os.path.basename(ref_path) if ref_path else "model's choice"
    print(f"\n🎙️ Job {JOB['id']} | ~{sum(d for _, d in plan):.1f} s of audio in {len(plan)} clip(s) | {steps} steps | "
          f"seed {seed} | voice: {voice}")
    sys.stdout.flush()

    ref_latent = None
    if ref_path:
        report(0.0, "Encoding the voice reference...")
        BAR.start("ref", "Voice reference")
        ref_latent = encode_reference(ref_path, float(ref_dur))
        BAR.finish()

    total_steps = steps * len(plan)
    wav, sr = None, None
    for idx, (text, dur) in enumerate(plan):
        label = "Denoising" if len(plan) == 1 else f"Clip {idx + 1}/{len(plan)}"
        tag = "" if len(plan) == 1 else f"Clip {idx + 1}/{len(plan)} · "

        def on_step(done, total, idx=idx, tag=tag):
            BAR.update(done)
            report(0.97 * (idx * steps + done) / total_steps, f"{tag}step {done}/{total}")

        report(0.97 * idx * steps / total_steps, f"{tag}encoding the prompt...")
        BAR.start("denoise", label, total=steps, unit="steps")
        try:
            clip, clip_sr = generate_clip(text, ref_latent, cfg, stg, dur, seed, steps, on_step)
        except BaseException:
            BAR.abort("stopped" if CANCEL.is_set() else "failed")
            raise
        BAR.finish(f"→ {clip.shape[-1] / clip_sr:.1f} s audio")
        clip = clip.cpu()
        if wav is None:
            wav, sr = clip, clip_sr
        else:
            if clip.shape[0] != wav.shape[0]:
                if clip.shape[0] == 1:
                    clip = clip.repeat(wav.shape[0], 1)
                elif wav.shape[0] == 1:
                    wav = wav.repeat(clip.shape[0], 1)
            wav = _equal_power_crossfade(wav, clip, sr, fade_ms=float(crossfade_ms))

    report(0.98, "Watermarking + saving...")
    if WATERMARKER is not None:
        try:
            mono = wav.mean(dim=0).numpy() if wav.shape[0] > 1 else wav[0].numpy()
            marked = np.asarray(WATERMARKER.apply_watermark(mono, sample_rate=sr), dtype=np.float32)
            marked = torch.from_numpy(marked).unsqueeze(0)
            wav = marked if wav.shape[0] == 1 else marked.repeat(wav.shape[0], 1)
        except Exception as e:
            print(f"  ⚠️ Watermark skipped ({e})")
    path = os.path.join(OUTPUT_DIR, f"dramabox_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed}.wav")
    sf.write(path, np.clip(wav.numpy().T, -1.0, 1.0), sr, subtype="PCM_16")
    elapsed = time.time() - t0
    audio_s = wav.shape[-1] / sr
    print(f"  💾 {audio_s:.1f} s of audio in {_fmt_secs(elapsed)} ({audio_s / max(elapsed, 1e-6):.1f}x real time) → {path}")
    sys.stdout.flush()
    status = (f"✅ {audio_s:.1f} s of audio in {elapsed:.1f} s ({audio_s / max(elapsed, 1e-6):.1f}x real time) · "
              f"seed {seed} · {len(plan)} clip(s) · {steps} steps\n💾 {path}")
    return path, seed, status


# ==== Generation as a background job ====
# The Generate button only starts a job and returns. The page asks for the job's state every second with short,
# independent requests (queue=False: plain HTTP, no event stream), so a dropped tunnel connection or a reloaded page
# never loses the result.
JOB = {"id": 0, "done": 0, "running": False, "audio": None, "seed": "", "status": "Ready."}


def _job_worker(job_id, args):
    try:
        with torch.cuda.device(DEV0):
            audio, seed, status = generate_audio(*args)
    except Cancelled:
        audio, seed, status = None, "", "⏹️ Stopped."
    except Exception as e:
        import traceback
        traceback.print_exc()
        audio, seed, status = None, "", f"❌ Error: {e}"
    finally:
        CANCEL.clear()
        gc.collect()
        for i in range(N_GPU):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()
    JOB.update(running=False, done=job_id, audio=audio, seed=str(seed), status=status)
    PROGRESS.update(active=False, finished=time.time(), fraction=1.0 if audio else PROGRESS["fraction"],
                    audio=os.path.basename(audio) if audio else None, message=status.splitlines()[0])


VOICE_NONE = "🎲 None (the model picks a voice)"
VOICE_UPLOAD = "📁 My own recording (upload below)"


def _voice_label(filename):
    stem = os.path.splitext(filename)[0]
    gender, _, name = stem.partition("_")
    return f"{'👩' if gender == 'female' else '👨'} {name.replace('_', ' ').title()} ({gender})"


VOICES = {_voice_label(f): os.path.join(VOICES_DIR, f) for f in sorted(os.listdir(VOICES_DIR))
          if f.lower().endswith((".wav", ".mp3", ".flac", ".ogg"))} if os.path.isdir(VOICES_DIR) else {}
STEP_PRESETS = {"💎 Quality - 30 steps (official)": 30, "⚖️ Balanced - 20 steps": 20, "⚡ Draft - 12 steps": 12}


def start_generation(prompt, voice_choice, audio_ref, preset, custom_steps, cfg, stg, dur_mult, gen_dur, ref_dur, seed,
                     max_chunk, target_chunk, crossfade_ms):
    if JOB["running"]:
        return gr.skip(), "⏳ Audio is already being generated. It appears here as soon as it is ready."
    if not prompt or not prompt.strip():
        return gr.skip(), "❌ The prompt is empty."
    # An uploaded / recorded / library clip in the audio box wins; otherwise the library voice picked in the dropdown
    ref_path = audio_ref if audio_ref and os.path.exists(str(audio_ref)) else VOICES.get(voice_choice)
    steps = int(custom_steps) if custom_steps and int(custom_steps) > 0 else STEP_PRESETS.get(preset, 30)
    args = (prompt, ref_path, steps, cfg, stg, dur_mult, gen_dur, ref_dur, seed, max_chunk, target_chunk, crossfade_ms)
    JOB.update(id=JOB["id"] + 1, running=True)
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, audio=None,
                    message="")
    threading.Thread(target=_job_worker, args=(JOB["id"], args), daemon=True).start()
    return None, "🚀 Generating... the progress bar above updates live, and the audio appears here when it is ready."


def _progress_card(done=False):
    p = PROGRESS
    end = p["finished"] or time.time()
    elapsed = end - p["started"] if p["started"] else 0.0
    frac = max(0.0, min(1.0, p["fraction"]))
    if done:
        ok = bool(p["audio"])
        color = "linear-gradient(135deg,#10b981,#059669)" if ok else "linear-gradient(135deg,#ef4444,#b91c1c)"
        link = (' · <a href="aiq_audio?t=%d" target="_blank" style="color:#667eea;font-weight:600">▶ open / download</a>'
                % int(time.time())) if ok else ""
        text = (p["message"] or ("Done" if ok else "Stopped")) + link
        frac = 1.0 if ok else frac
    else:
        color = "linear-gradient(135deg,#667eea 0%,#764ba2 100%)"
        eta = ""
        if 0.03 < frac < 0.97 and elapsed > 2:
            eta = f" · ETA {_fmt_secs(elapsed * (1 - frac) / frac)}"
        text = f"⏳ {p['text'] or 'Working...'} · {_fmt_secs(elapsed)}{eta}"
    return (f'<div class="aiq-progress"><div class="aiq-progress-top"><span>{text}</span>'
            f'<span class="aiq-pct">{frac * 100:.0f}%</span></div><div class="aiq-track">'
            f'<div class="aiq-fill" style="width:{frac * 100:.1f}%;background:{color}"></div></div></div>')


IDLE_CARD = ('<div class="aiq-progress"><div class="aiq-progress-top"><span>💤 Ready. Write a scene and press Generate.'
             '</span><span class="aiq-pct"></span></div><div class="aiq-track"><div class="aiq-fill" '
             'style="width:0%"></div></div></div>')


def poll_job(seen):
    """Every second: live progress while a job runs; the result once per page when it is done."""
    if JOB["running"]:
        return _progress_card(), gr.skip(), gr.skip(), gr.skip(), seen
    if JOB["done"] and JOB["done"] != seen:
        return _progress_card(done=True), JOB["audio"], JOB["seed"], JOB["status"], JOB["done"]
    return gr.skip(), gr.skip(), gr.skip(), gr.skip(), seen


def stop_generation():
    if not JOB["running"]:
        return "Nothing is running."
    CANCEL.set()
    return "⏹️ Stopping after the current step..."


def _latest_audio():
    files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.wav")), key=os.path.getmtime)
    return files[-1] if files else None


def load_latest_audio():
    path = _latest_audio()
    return (path, f"📁 Latest audio: {path}") if path else (None, f"No audio in {OUTPUT_DIR} yet.")


def pick_voice(choice):
    if choice in VOICES:
        return gr.update(value=VOICES[choice])
    if choice == VOICE_UPLOAD:
        return gr.skip()  # keep whatever is in the audio box
    return gr.update(value=None)


# ==== Gradio UI (AIQUEST Academy branding) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
.chunk-preview { font-size: 12px !important; color: #6b7280 !important; padding: 6px 12px !important; border-left: 3px solid #764ba2; background: rgba(102,126,234,0.06); border-radius: 4px; min-height: 0 !important; }
.chunk-preview p { margin: 0 !important; }
.aiq-progress { padding: 12px 14px; border-radius: 12px; background: rgba(102,126,234,0.07); border: 1px solid rgba(102,126,234,0.25); }
.aiq-progress-top { display: flex; justify-content: space-between; gap: 12px; font-size: 13px; margin-bottom: 8px; }
.aiq-pct { font-weight: 700; color: #764ba2; }
.aiq-track { height: 10px; background: rgba(118,75,162,0.15); border-radius: 6px; overflow: hidden; }
.aiq-fill { height: 100%; border-radius: 6px; transition: width .8s ease; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎭 DramaBox - Expressive TTS</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
_mode = ("prompt + STG passes on GPU 0, negative-prompt pass on GPU 1" if DIT1 is not None
         else "all guidance passes on GPU 0")
STATUS_HTML = (f'<div class="status-line">⚡ FP16 DiT on {N_GPU} x {_props[0].name} &nbsp;|&nbsp; {_mode} '
               f'&nbsp;|&nbsp; Gemma-3 12B 4-bit, cached prompts &nbsp;|&nbsp; 48 kHz stereo, Perth watermark</div>')
FOOTER_HTML = ('<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · '
               '© All rights reserved</div>')

_V = {os.path.basename(p): label for label, p in VOICES.items()}
EXAMPLES = [
    ['A shadowy villain speaks with cold menace, "You have entered my domain, mortal." He chuckles darkly, '
     '"Such arrogance will be your undoing." His voice rises with fury, "Kneel, or be destroyed where you stand!"',
     _V.get("male_harvey_keitel.mp3", VOICE_NONE)],
    ['A talk show host gasps with shock, "No! You did NOT just say that!" He bursts into uncontrollable laughter, '
     '"Hahaha! Oh my god, oh my god!" He wheezes, "I cannot, I literally cannot breathe right now!"',
     _V.get("male_conan.mp3", VOICE_NONE)],
    ['A woman speaks tenderly, "It has been a long day, my love." She whispers, "Close your eyes. I am right here." '
     'She hums quietly, "Mmmm-mmm. Sleep now."', _V.get("female_shadowheart.wav", VOICE_NONE)],
    ['A radio host clears his throat, "Excuse me, pardon that." He settles into a warm, professional tone, '
     '"Good evening everyone, and welcome back to the show. We have got a wonderful lineup tonight."',
     _V.get("male_old_movie.wav", VOICE_NONE)],
    ['A playful girl already mid-giggle, "Hehehe, oh my gosh you should see your face!" She gasps for air between '
     'giggles, "Oh my, hehe, oh my, I cannot stop!" She tries to compose herself, "Ahhhhh okay okay okay, I will stop, '
     'I promise."', _V.get("female_american.wav", VOICE_NONE)],
]

print("🖥️ Building the UI...")
sys.stdout.flush()
with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="DramaBox Expressive TTS | AIQUEST Academy",
               analytics_enabled=False) as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)
    gr.Markdown("💡 **Tips:** put dialogue inside `\"double quotes\"` and scene directions outside. Sounds like "
                "`\"Hahaha\"`, `\"Mmmm\"`, `\"Ugh\"` go inside the quotes; actions like *She sighs.* or *He clears his "
                "throat.* go outside. Long texts are split at sentence ends and joined with a crossfade.")
    with gr.Row():
        with gr.Column(scale=3):
            prompt_box = gr.Textbox(label="🎬 Scene prompt", lines=6, placeholder=EXAMPLES[0][0])
            chunk_preview = gr.Markdown("", elem_classes=["chunk-preview"])
            voice_dd = gr.Dropdown([VOICE_NONE, VOICE_UPLOAD] + list(VOICES), value=VOICE_NONE,
                                   label="🎙️ Voice", info="A built-in voice, your own recording, or none")
            audio_ref = gr.Audio(label="Voice reference (10+ s of clean speech works best)", type="filepath")
        with gr.Column(scale=2):
            preset = gr.Dropdown(list(STEP_PRESETS), value=next(iter(STEP_PRESETS)), label="💎 Quality / speed",
                                 info="Time grows with the number of steps")
            seed_input = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = random)")
            with gr.Accordion("⚙️ Advanced", open=False):
                cfg_slider = gr.Slider(1.0, 10.0, value=2.5, step=0.5, label="CFG scale")
                stg_slider = gr.Slider(0.0, 5.0, value=1.5, step=0.5, label="STG scale")
                custom_steps = gr.Slider(0, 60, value=0, step=1, label="Custom steps (0 = use the preset)")
                dur_slider = gr.Slider(0.8, 2.0, value=1.1, step=0.05, label="Duration multiplier (when length = auto)")
                gen_dur_slider = gr.Slider(0.0, 600.0, value=0.0, step=1.0,
                                           label="Target length in s (0 = auto from the text)")
                ref_dur_slider = gr.Slider(3.0, 30.0, value=10.0, step=1.0, label="Voice reference length used (s)")
            with gr.Accordion("📚 Long texts (chunking)", open=False):
                max_chunk_slider = gr.Slider(20.0, 60.0, value=45.0, step=1.0, label="Max chunk length (s)")
                target_chunk_slider = gr.Slider(15.0, 50.0, value=37.0, step=1.0, label="Target chunk length (s)")
                crossfade_slider = gr.Slider(0.0, 250.0, value=50.0, step=5.0, label="Crossfade (ms)")

    with gr.Row():
        gen_btn = gr.Button("🎭 Generate Speech", variant="primary", size="lg")
        stop_btn = gr.Button("🛑 Stop", size="lg", elem_id="stop-btn")
        clear_btn = gr.Button("🗑️ Clear", size="lg", elem_id="clear-btn")
    progress_html = gr.HTML(IDLE_CARD)
    audio_out = gr.Audio(label="🔊 Generated audio", type="filepath", interactive=False)
    with gr.Row():
        used_seed = gr.Textbox(label="🎲 Seed used", interactive=False, scale=1)
        status_box = gr.Textbox(label="ℹ️ Status", lines=2, interactive=False, scale=3)
    latest_btn = gr.Button("📂 Load latest audio (if the page lost its connection)")
    seen_job = gr.State(0)
    poll_timer = gr.Timer(1.0)

    gr.Examples(EXAMPLES, inputs=[prompt_box, voice_dd], label="✨ Example scenes (click, then Generate)")
    gr.HTML(FOOTER_HTML)

    for trigger in (prompt_box, dur_slider, max_chunk_slider, target_chunk_slider):
        trigger.change(preview_chunks, inputs=[prompt_box, dur_slider, max_chunk_slider, target_chunk_slider],
                       outputs=chunk_preview, queue=False, show_progress="hidden")
    voice_dd.change(pick_voice, inputs=voice_dd, outputs=audio_ref, queue=False, show_progress="hidden")
    for _event in (audio_ref.upload, audio_ref.stop_recording):  # an own clip: show it in the dropdown
        _event(lambda: VOICE_UPLOAD, outputs=voice_dd, queue=False, show_progress="hidden")
    gen_btn.click(start_generation,
                  inputs=[prompt_box, voice_dd, audio_ref, preset, custom_steps, cfg_slider, stg_slider, dur_slider,
                          gen_dur_slider, ref_dur_slider, seed_input, max_chunk_slider, target_chunk_slider,
                          crossfade_slider],
                  outputs=[audio_out, status_box], queue=False)
    poll_timer.tick(poll_job, inputs=seen_job, outputs=[progress_html, audio_out, used_seed, status_box, seen_job],
                    queue=False, show_progress="hidden")
    stop_btn.click(stop_generation, outputs=status_box, queue=False)
    latest_btn.click(load_latest_audio, outputs=[audio_out, status_box], queue=False)
    clear_btn.click(lambda: ("", VOICE_NONE, None, None, "", "Ready.", -1, IDLE_CARD),
                    outputs=[prompt_box, voice_dd, audio_ref, audio_out, used_seed, status_box, seed_input,
                             progress_html], queue=False)


# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
# cloudflared tries QUIC (UDP) first, which Kaggle's network often blocks: hence --protocol http2. A browser that asks
# for the random *.trycloudflare.com name before it exists caches the "not found" answer for minutes, so the link is
# only shown as ready once cloudflared has registered AND the name resolves.
def _dns_published(host):
    """Ask Cloudflare's DNS-over-HTTPS directly (no local resolver caches a negative answer). True / False / None."""
    try:
        req = urllib.request.Request(f"https://cloudflare-dns.com/dns-query?name={host}&type=A",
                                     headers={"accept": "application/dns-json"})
        return bool(json.loads(urllib.request.urlopen(req, timeout=5).read()).get("Answer"))
    except Exception:
        return None


def start_cloudflare_tunnel(port):
    import stat
    found = {"url": None, "registered": False, "ready": False}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                                   "--protocol", "http2"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        import atexit
        atexit.register(tunnel.kill)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)
            if match and not found["url"]:
                found["url"] = match.group(0)
            if "Registered tunnel connection" in line:
                found["registered"] = True

    def _wait_ready():
        deadline, registered_at = time.time() + 300, None
        while time.time() < deadline:
            if found["registered"] and registered_at is None:
                registered_at = time.time()
            published = _dns_published(found["url"].split("//", 1)[1]) if found["url"] and registered_at else False
            if published or (published is None and time.time() - registered_at > 15):
                found["ready"] = True
                if PRINTED_LINKS.is_set():
                    print(f"\n✅ Cloudflare link is ready: {found['url']}\n")
                    sys.stdout.flush()
                return
            time.sleep(3)
        print("⚠️ The Cloudflare link did not come up within 5 minutes; use the Gradio share link.")

    threading.Thread(target=_watch, daemon=True).start()
    threading.Thread(target=_wait_ready, daemon=True).start()
    return found


PRINTED_LINKS = threading.Event()
print("🌐 Starting the public links (Cloudflare + Gradio share)...")
sys.stdout.flush()
cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20)
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
            prevent_thread_lock=True, quiet=True, ssr_mode=False, show_api=False, max_threads=8,
            allowed_paths=[OUTPUT_DIR, VOICES_DIR])
from fastapi.responses import FileResponse, JSONResponse
from fastapi.routing import APIRoute


def _progress_route():
    end = PROGRESS["finished"] or time.time()
    elapsed = end - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})


def _audio_route():
    """The newest audio as a plain file (plays in the browser, with a download option): no Gradio involved."""
    path = _latest_audio()
    if not path:
        return JSONResponse({"error": "no audio yet"}, status_code=404)
    name = os.path.basename(path)
    return FileResponse(path, media_type="audio/wav",
                        headers={"Content-Disposition": f'inline; filename="{name}"', "Cache-Control": "no-store"})


for _path, _route in (("/aiq_progress", _progress_route), ("/aiq_audio", _audio_route)):
    demo.app.router.routes.insert(0, APIRoute(_path, _route, methods=["GET"]))  # ahead of Gradio's routes
for _ in range(60):  # give the tunnel up to ~30 s before printing the links
    if cf["ready"]:
        break
    time.sleep(0.5)

print("\n" + "=" * 70)
print("🎭 DramaBox is LIVE!")
print("=" * 70)
if cf["ready"]:
    print(f"🌐 Cloudflare tunnel: {cf['url']}")
else:
    print(f"🌐 Cloudflare tunnel: ⏳ warming up{' (' + cf['url'] + ')' if cf['url'] else ''} - wait for the "
          f"'✅ Cloudflare link is ready' line before opening it")
print(f"🔗 Gradio share:      {getattr(demo, 'share_url', None) or '(unavailable, use the Cloudflare link)'}")
print(f"💾 Audio is saved to {OUTPUT_DIR} (Kaggle Output panel)")
print(f"🔊 Latest audio without the UI: <link>/aiq_audio")
print("=" * 70 + "\n")
PRINTED_LINKS.set()
sys.stdout.flush()
while True:  # keep the server process alive; generation progress streams to the notebook cell
    time.sleep(3600)
'''

with open("run_dramabox.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)
print("✅ run_dramabox.py written.")

if os.path.exists("dramabox_settings.json"):  # Step 1's paths, for a Step 2 re-run after a kernel restart
    with open("dramabox_settings.json") as f:
        for key, value in json.load(f).items():
            os.environ.setdefault(key, value)

# Library noise that is not an error; everything else (including tracebacks) is shown.
IGNORE = ("absl::InitializeLog", "Unable to register cu", "computation placer already registered", "MessageFactory",
          "GetPrototype", "Attempting to register factory", "UserWarning", "FutureWarning", "DeprecationWarning",
          "warnings.warn", "`torch_dtype` is deprecated", "Using a slow image processor", "TypedStorage is deprecated",
          "Sliding Window Attention is enabled")
env = dict(os.environ, PYTHONWARNINGS="ignore", TQDM_DISABLE="1", TRANSFORMERS_VERBOSITY="error",
           HF_HUB_DISABLE_PROGRESS_BARS="1", BITSANDBYTES_NOWELCOME="1", GRADIO_ANALYTICS_ENABLED="False",
           TOKENIZERS_PARALLELISM="false", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

print("🚀 Starting the DramaBox engine...\n")
import psutil


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~20 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_dramabox.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()

# Step 1's downloads leave freed buffers in this kernel process: hand them back to the OS before the engine starts.
import ctypes
import gc
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass


def _die_with_kernel():
    # Linux: the engine gets SIGKILL when this notebook kernel dies or restarts (no orphan holding the GPUs).
    try:
        import signal
        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGKILL)  # PR_SET_PDEATHSIG
    except Exception:
        pass


proc = subprocess.Popen([sys.executable, "-u", "run_dramabox.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        env=env, preexec_fn=_die_with_kernel)
import threading
_peak = {"ram": 0.0, "engine": 0.0}


def _watch_ram():  # peak RAM while the engine runs, reported if it dies
    engine = psutil.Process(proc.pid)
    while proc.poll() is None:
        try:
            vm = psutil.virtual_memory()
            _peak["ram"] = max(_peak["ram"], (vm.total - vm.available) / 1024**3)
            _peak["engine"] = max(_peak["engine"], engine.memory_info().rss / 1024**3)
        except Exception:
            pass
        time.sleep(0.5)


threading.Thread(target=_watch_ram, daemon=True).start()
# Read raw bytes so carriage returns survive: the engine redraws its live progress line in place, and a text-mode
# pipe would turn every redraw into a new line.
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()  # text after the last line break / carriage return
        out = "".join(part for part in parts if not any(pattern in part for pattern in IGNORE))
        if pending and not any(pattern in pending for pattern in IGNORE):
            out, pending = out + pending, ""  # show the live progress line right away
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        import signal
        how = f"killed by {signal.Signals(-rc).name}" if rc < 0 else f"exit code {rc}"
        hint = (" - the system killed it, which almost always means it ran out of RAM" if rc == -9 else
                " - a crash inside a native library (see the traceback above)" if rc in (-11, -6) else "")
        print(f"\n❌ The engine stopped ({how}){hint}.\n   Peak RAM while it ran: {_peak['ram']:.1f} GB in use of "
              f"{psutil.virtual_memory().total / 1024**3:.1f} GB (engine process {_peak['engine']:.1f} GB).")
except KeyboardInterrupt:
    proc.kill()  # frees the GPUs and RAM right away
    proc.wait()
    print("🛑 Engine stopped. Re-run this cell to start it again.")

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---